In [4]:
import pandas as pd
df = pd.read_parquet(r"D:\work\trade\data_center\storage\market_data\stock_daily\002487.SZ.parquet")
df

,open,high,low,close,volume,amount,code,adj_factor,limit_up,limit_down
datetime,,,,,,,,,,
2014-01-02,6.03,6.03,5.89,5.95,34530,2.056605e+07,002487.SZ,1.0,NaN,NaN
2014-01-03,5.89,5.95,5.80,5.83,32622,1.905088e+07,002487.SZ,1.0,6.55,5.35
2014-01-06,5.80,5.80,5.65,5.65,31166,1.780656e+07,002487.SZ,1.0,6.41,5.25
2014-01-07,5.61,5.72,5.58,5.67,21171,1.198401e+07,002487.SZ,1.0,6.22,5.09
2014-01-08,5.66,5.75,5.66,5.69,23528,1.344472e+07,002487.SZ,1.0,6.24,5.10
...,...,...,...,...,...,...,...,...,...,...
2025-12-10,56.79,57.72,56.47,56.71,114807,6.540983e+08,002487.SZ,1.0,62.74,51.34
2025-12-11,57.20,58.80,56.73,57.61,172055,9.968762e+08,002487.SZ,1.0,62.38,51.04
2025-12-12,57.36,58.10,56.80,57.49,141372,8.132906e+08,002487.SZ,1.0,63.37,51.85


In [3]:
df.columns

Index(['open', 'high', 'low', 'close', 'volume', 'amount', 'code',
       'adj_factor', 'limit_up', 'limit_down'],
      dtype='object')

In [4]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm  # 用于显示进度条

def process_stock_data(folder_path):
    # 1. 获取文件夹下所有的 parquet 文件
    file_pattern = os.path.join(folder_path, "*.parquet")
    file_list = glob.glob(file_pattern)
    
    if not file_list:
        print("未找到任何 .parquet 文件，请检查路径。")
        return None

    print(f"共发现 {len(file_list)} 个股票文件，开始处理...")

    processed_dfs = []

    # 2. 遍历处理每个文件
    for file_path in tqdm(file_list, desc="Processing Files"):
        try:
            # 读取 Parquet
            df = pd.read_parquet(file_path)
            
            # 确保 Index 是 datetime 格式
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
            
            # 数据清洗：如果有 NaN 的 close，视为无效数据，去除
            df = df.dropna(subset=['close'])
            
            if df.empty:
                continue

            # --- 计算所需的中间指标 ---
            
            # 1. 计算涨跌幅 (今天收盘 - 昨天收盘) / 昨天收盘
            # 注意：这里假设数据是按时间排序的
            df['pct_chg'] = df['close'].pct_change()
            
            # 2. 判断涨跌停
            # 浮点数比较建议加上一个微小的误差值 (epsilon)，或者使用 np.isclose
            # 这里为了性能和通用性，使用 >= (limit_up - 0.01) 这种逻辑，防止精度丢失导致相等判断失败
            epsilon = 1e-4
            
            # 逻辑：收盘价 >= 涨停价
            df['is_limit_up'] = (df['close'] >= (df['limit_up'] - epsilon)).astype(int)
            
            # 逻辑：收盘价 <= 跌停价
            df['is_limit_down'] = (df['close'] <= (df['limit_down'] + epsilon)).astype(int)
            
            # 3. 标记这只股票当天是有效的 (用于统计个股总数)
            df['is_valid'] = 1
            
            # 4. 只保留需要的列以节省内存
            # 我们只需要：日期(Index), is_limit_up, is_limit_down, pct_chg, is_valid
            mini_df = df[['is_limit_up', 'is_limit_down', 'pct_chg', 'is_valid']]
            
            processed_dfs.append(mini_df)
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    print("正在合并数据并进行每日统计...")
    
    # 3. 合并所有股票的轻量级数据
    if not processed_dfs:
        return pd.DataFrame()
        
    all_stocks = pd.concat(processed_dfs)

    # 4. 按日期 (Index) Groupby 聚合
    # sum: 用于计算涨停数、跌停数、有效股票总数
    # mean: 用于计算平均涨跌幅 (注意：pct_chg 有 NaN 的天数会自动被 mean 忽略，符合逻辑)
    
    daily_stats = all_stocks.groupby(all_stocks.index).agg({
        'is_limit_up': 'sum',
        'is_limit_down': 'sum',
        'is_valid': 'sum',     # 当天有多少只股票有数据
        'pct_chg': 'mean'      # 第六列：当天平均涨跌幅
    })

    # 5. 计算比例
    # 第四列：涨停比例
    daily_stats['limit_up_ratio'] = daily_stats['is_limit_up'] / daily_stats['is_valid']
    
    # 第五列：跌停比例
    daily_stats['limit_down_ratio'] = daily_stats['is_limit_down'] / daily_stats['is_valid']

    # 6. 整理最终的 DataFrame 格式
    # 重命名列以符合你的要求
    final_df = daily_stats.rename(columns={
        'is_limit_up': 'limit_up_count',       # 第二列：涨停个数
        'is_limit_down': 'limit_down_count',   # 第三列：跌停个数
        'pct_chg': 'avg_pct_change',           # 第六列：当天平均涨跌幅
        # 'is_valid' 不需要输出，或者可以作为参考保留
    })

    # 调整列顺序
    final_df = final_df[[
        'limit_up_count', 
        'limit_down_count', 
        'limit_up_ratio', 
        'limit_down_ratio', 
        'avg_pct_change'
    ]]
    
    # 如果需要 Date 作为第一列而不是 Index，可以 reset_index
    final_df = final_df.reset_index().rename(columns={'index': 'date'})

    return final_df

# --- 执行代码 ---
# 请修改为你的实际路径
folder_path = r"D:\work\trade\data_center\storage\market_data\stock_daily"

# 运行处理
df_market_sentiment = process_stock_data(folder_path)

# 查看结果
if df_market_sentiment is not None:
    print(df_market_sentiment.head())
    
    # 输出到 CSV
    output_path = "market_daily_stats.csv"
    df_market_sentiment.to_csv(output_path, index=False)
    print(f"文件已保存至: {output_path}")

共发现 5175 个股票文件，开始处理...


Processing Files: 100%|██████████| 5175/5175 [00:37<00:00, 136.47it/s]


正在合并数据并进行每日统计...
    datetime  limit_up_count  limit_down_count  limit_up_ratio  \
0 2014-01-02               0                 0        0.000000   
1 2014-01-03              13                 0        0.006007   
2 2014-01-06               6                 6        0.002765   
3 2014-01-07              10                 3        0.004598   
4 2014-01-08              10                 0        0.004598   

   limit_down_ratio  avg_pct_change  
0          0.000000             NaN  
1          0.000000       -0.005610  
2          0.002765       -0.029402  
3          0.001379        0.007098  
4          0.000000        0.003988  
文件已保存至: market_daily_stats.csv


In [6]:
df_market_sentiment.to_csv(r"D:\work\data2\market_daily_stats.csv", index=False)

In [7]:
import pandas as pd
import numpy as np
import os
import glob
import re
from tqdm import tqdm

# ================= 配置区 =================
# 请确保路径没有多余空格
STOCK_DATA_PATH = r"D:\work\trade\data_center\storage\market_data\stock_daily"
MARKET_DATA_PATH = r"D:\work\data2\market_daily_stats.csv"
OUTPUT_PATH = r"D:\work\data2\train_data.csv"

# ================= 1. 核心逻辑函数 =================

def load_market_data(path):
    """
    读取并预处理大盘数据，增加了对列名的容错处理
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"未找到大盘数据文件: {path}")
    
    # 读取 CSV
    df_market = pd.read_csv(path)
    
    # --- 自动寻找日期列 ---
    # 1. 如果本来就有 'date' 列
    if 'date' in df_market.columns:
        date_col = 'date'
    # 2. 如果没有 'date'，通常第一列就是日期（可能是 'index', 'Unnamed: 0', 'datetime' 等）
    else:
        # 获取第一列的列名
        date_col = df_market.columns[0]
        print(f"提示：未在表头找到 'date'，自动将第一列 '{date_col}' 识别为日期列。")
    
    # 统一重命名为 'date' 以便后续处理
    df_market.rename(columns={date_col: 'date'}, inplace=True)
    
    # 转换格式并设为索引
    try:
        df_market['date'] = pd.to_datetime(df_market['date'])
        df_market.set_index('date', inplace=True)
    except Exception as e:
        raise ValueError(f"日期列转换失败，请检查 CSV 格式。错误信息: {e}")
        
    return df_market

def calculate_technical_features(df):
    """计算训练数据 X (形态、波动) 和 Y (标签)"""
    # --- B. 计算 X (个股特征) ---
    df['ma5'] = df['close'].rolling(window=5).mean()
    df['ma_vol_5'] = df['volume'].rolling(window=5).mean()
    df['pre_close'] = df['close'].shift(1)
    
    # 1. 形态强弱类
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
    df['body_size'] = abs(df['close'] - df['open']) / df['open']
    df['ma5_bias'] = (df['close'] - df['ma5']) / df['ma5']
    df['vol_ratio'] = df['volume'] / df['ma_vol_5'].replace(0, np.nan)
    
    # 2. 波动类
    df['high_low_ratio'] = (df['high'] - df['low']) / df['pre_close']

    # --- C. 计算 Y (标签: T+1的表现) ---
    next_close = df['close'].shift(-1)
    next_open = df['open'].shift(-1)
    next_high = df['high'].shift(-1)
    
    # 逻辑：(收阳) AND (冲高 > 3%)
    cond_1 = next_close > next_open
    cond_2 = (next_high - next_open) / next_open > 0.03
    
    df['label'] = (cond_1 & cond_2).astype(int)
    
    return df

def apply_strategy_filter(df):
    """
    N字反转筛选 + 过热过滤
    """
    # 1. 基础计算
    # 使用 1e-4 误差处理浮点数比较
    df['is_limit_up'] = (df['close'] >= (df['limit_up'] - 1e-4)).astype(int)
    # 过去20天涨停数（不含今天，shift(1)）
    df['recent_limit_ups_20'] = df['is_limit_up'].shift(1).rolling(window=60).sum()
    
    # 2. 辅助列 (L1=昨天涨停...)
    df['L1'] = df['is_limit_up'].shift(1) == 1
    df['L2'] = df['is_limit_up'].shift(2) == 1
    df['L3'] = df['is_limit_up'].shift(3) == 1
    
    df['open_T1'] = df['open'].shift(1)
    df['open_T2'] = df['open'].shift(2)
    df['open_T3'] = df['open'].shift(3)
    
    # 3. 筛选 Mask
    
    # 过热逻辑 (过去20天涨停 < 5)
    not_overheated = (df['recent_limit_ups_20'] < 6)
    
    # 场景 1: T-1涨停，T未涨停
    past_counts_1 = df['is_limit_up'].shift(2).rolling(6).sum()
    case_1 = (
        (df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_1 < 2) &
        (df['close'] >= df['open_T1'])
    )

    # 场景 2: T-2涨停，T-1, T未涨停
    past_counts_2 = df['is_limit_up'].shift(3).rolling(6).sum()
    case_2 = (
        (df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_2 < 2) &
        (df['close'] >= df['open_T2'])
    )

    # 场景 3: T-3涨停，T-2, T-1, T未涨停
    past_counts_3 = df['is_limit_up'].shift(4).rolling(6).sum()
    case_3 = (
        (df['L3']) & (~df['L2']) & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_counts_3 < 2) &
        (df['close'] >= df['open_T3'])
    )

    # 综合筛选
    # 必须符合: (任意一种形态) AND (不过热) AND (是有效交易日)
    final_mask = (case_1 | case_2 | case_3) & not_overheated & (df['is_valid'] == 1)
    
    return df[final_mask].copy()

# ================= 2. 主流程 =================

def main():
    print("1. 正在加载大盘数据...")
    try:
        df_market = load_market_data(MARKET_DATA_PATH)
        print(f"大盘数据加载成功，时间跨度: {df_market.index.min()} 至 {df_market.index.max()}")
        market_cols = df_market.columns.tolist() 
    except Exception as e:
        print(f"加载大盘数据失败: {e}")
        # 如果大盘数据挂了，这里直接return，否则后面merge会报错
        return

    print("2. 扫描并筛选股票文件...")
    all_files = glob.glob(os.path.join(STOCK_DATA_PATH, "*.parquet"))
    
    # 筛选 00 或 60 开头的股票 (排除 300, 688 等)
    target_pattern = re.compile(r'^(00|60)\d+') 
    
    valid_files = []
    for f in all_files:
        filename = os.path.basename(f)
        code = filename.split('.')[0]
        if target_pattern.match(code):
            valid_files.append(f)
            
    print(f"符合 60/00 开头的股票文件数: {len(valid_files)}")
    
    result_list = []
    
    # 3. 遍历处理
    for file_path in tqdm(valid_files, desc="Processing Stocks"):
        try:
            df = pd.read_parquet(file_path)
            
            # 基础清洗
            if df.empty: continue
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)
            df = df.sort_index()
            # 标记 close 为 NaN 的行为无效
            df = df.dropna(subset=['close'])
            df['is_valid'] = 1 

            # A. 计算个股特征和 Label (此时还是全量日期)
            df = calculate_technical_features(df)
            
            # B. 策略筛选 (此时只剩下符合条件的日期)
            selected_df = apply_strategy_filter(df)
            
            if not selected_df.empty:
                # C. 合并大盘数据 (Left Join, 按 index 也就是 date 自动对齐)
                combined_df = selected_df.join(df_market, how='left')
                
                # 补充 Meta 信息
                combined_df['code'] = os.path.basename(file_path).split('.')[0]
                combined_df['date'] = combined_df.index
                
                # D. 提取需要的列
                cols_to_keep = [
                    'date', 'code',
                    # 个股因子
                    'upper_shadow', 'body_size', 'ma5_bias', 'vol_ratio', 'high_low_ratio',
                    # 大盘因子 (自动添加所有大盘数据列)
                ] + market_cols + [
                    # 标签
                    'label'
                ]
                
                result_list.append(combined_df[cols_to_keep])
                
        except Exception as e:
            # 单个文件报错不中断整个流程
            # print(f"Error in {file_path}: {e}") 
            continue

    # 4. 汇总保存
    print("正在合并最终数据集...")
    if result_list:
        final_train_data = pd.concat(result_list, ignore_index=True)
        
        # 排序
        final_train_data = final_train_data.sort_values(by=['date', 'code'])
        
        # 清理无 Label 的数据 (通常是每只股票的最后一天)
        final_train_data = final_train_data.dropna(subset=['label'])
        
        print(f"最终生成训练样本数: {len(final_train_data)}")
        
        # 检查是否有列缺失
        print(f"包含特征列: {final_train_data.columns.tolist()}")
        
        # 创建目录并保存
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        final_train_data.to_csv(OUTPUT_PATH, index=False)
        print(f"全部完成！文件保存至: {OUTPUT_PATH}")
    else:
        print("未筛选出任何符合条件的样本。")

if __name__ == "__main__":
    main()

1. 正在加载大盘数据...
提示：未在表头找到 'date'，自动将第一列 'datetime' 识别为日期列。
大盘数据加载成功，时间跨度: 2014-01-02 00:00:00 至 2025-12-15 00:00:00
2. 扫描并筛选股票文件...
符合 60/00 开头的股票文件数: 3188


Processing Stocks: 100%|██████████| 3188/3188 [00:52<00:00, 61.19it/s]


正在合并最终数据集...
最终生成训练样本数: 170764
包含特征列: ['date', 'code', 'upper_shadow', 'body_size', 'ma5_bias', 'vol_ratio', 'high_low_ratio', 'limit_up_count', 'limit_down_count', 'limit_up_ratio', 'limit_down_ratio', 'avg_pct_change', 'label']
全部完成！文件保存至: D:\work\data2\train_data.csv


In [8]:
train = pd.read_csv(r"D:\work\data2\train_data.csv")
train['label'].value_counts()

label
0    110440
1     60324
Name: count, dtype: int64

In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ================= 配置区 =================
DATA_PATH = r"D:\work\data2\train_data.csv"
MODEL_DIR = r"D:\work\data"  # 模型保存目录

# --- 1. 时间窗口设置 (请根据你的数据实际时间修改) ---
# 格式: 'YYYY-MM-DD'
TRAIN_START = '2020-01-01'
TRAIN_END   = '2025-01-01'

VALID_START = '2025-01-01'
VALID_END   = '2025-12-31'

# --- 2. 特征与标签 ---
FEATURES = [
    'upper_shadow',      # 上影线
    'ma5_bias',          # 均线乖离
    'vol_ratio',         # 量比
    'limit_up_count',    # 大盘涨停数
    'avg_pct_change',    # 大盘平均涨跌幅
    'body_size',         # 实体大小
    'high_low_ratio'     # 振幅
]
LABEL = 'label'
DATE_COL = 'date'

# ================= 核心工具函数 =================

def load_data(path):
    print(f"1. 读取数据: {path} ...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    
    # 简单的缺失值填充 (树模型其实可以不填，但逻辑回归需要)
    df = df.dropna(subset=[LABEL]) # 标签为空必须删
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].mean())
    
    return df.sort_values(DATE_COL)

def get_dataset_by_date(df, start_date, end_date):
    """根据时间段截取数据"""
    mask = (df[DATE_COL] >= start_date) & (df[DATE_COL] <= end_date)
    return df[mask].copy()

def evaluate_top_k(df, model_name, pred_col='pred_score', top_n=5):
    """计算 Top 5 实战命中率"""
    daily_precisions = []
    
    for date, group in df.groupby(DATE_COL):
        if len(group) < 1: continue
        
        # 选分最高的 Top N
        top_picks = group.sort_values(by=pred_col, ascending=False).head(top_n)
        
        # 计算命中数
        hits = top_picks[LABEL].sum()
        # 实际选了多少只 (有可能那天只有3只票)
        actual_k = len(top_picks)
        
        if actual_k > 0:
            daily_precisions.append(hits / actual_k)
            
    if not daily_precisions:
        return 0.0
        
    return np.mean(daily_precisions)

# ================= 模型训练包装类 =================

class ModelTrainer:
    def __init__(self, train_df, valid_df):
        self.X_train = train_df[FEATURES]
        self.y_train = train_df[LABEL]
        self.X_valid = valid_df[FEATURES]
        self.y_valid = valid_df[LABEL]
        self.valid_df_raw = valid_df.copy() # 保留原始数据用于回测统计

    def run_lightgbm(self):
        print("\n--- Training LightGBM ---")
        train_data = lgb.Dataset(self.X_train, label=self.y_train)
        valid_data = lgb.Dataset(self.X_valid, label=self.y_valid, reference=train_data)
        
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'learning_rate': 0.05,
            'verbose': -1,
            'n_jobs': -1,
            'seed': 42
        }
        
        model = lgb.train(
            params, train_data, num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)] # 0表示不刷屏
        )
        return model, model.predict(self.X_valid)

    def run_xgboost(self):
            print("\n--- Training XGBoost ---")
            # 修改点 1: 将 early_stopping_rounds 放入初始化函数中
            model = xgb.XGBClassifier(
                objective='binary:logistic',
                eval_metric='auc',
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=5,
                n_jobs=-1,
                random_state=42,
                early_stopping_rounds=50  # <--- 从 fit() 移到这里
            )
            
            # 修改点 2: fit() 中去掉 early_stopping_rounds
            model.fit(
                self.X_train, self.y_train,
                eval_set=[(self.X_valid, self.y_valid)],
                verbose=False
                # early_stopping_rounds=50  <--- 这里删掉
            )
            # XGBoost predict_proba 返回 [class0_prob, class1_prob]
            return model, model.predict_proba(self.X_valid)[:, 1]

    def run_catboost(self):
        print("\n--- Training CatBoost ---")
        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=42,
            allow_writing_files=False
        )
        
        model.fit(
            self.X_train, self.y_train,
            eval_set=(self.X_valid, self.y_valid),
            early_stopping_rounds=50
        )
        return model, model.predict_proba(self.X_valid)[:, 1]

# ================= 主流程 =================

if __name__ == "__main__":
    # 1. 加载全量数据
    try:
        full_df = load_data(DATA_PATH)
    except Exception as e:
        print(e)
        exit()

    # 2. 按指定时间切分
    print(f"2. 切分数据集...")
    train_df = get_dataset_by_date(full_df, TRAIN_START, TRAIN_END)
    valid_df = get_dataset_by_date(full_df, VALID_START, VALID_END)
    
    print(f"   训练集 ({TRAIN_START} ~ {TRAIN_END}): {len(train_df)} 条")
    print(f"   验证集 ({VALID_START} ~ {VALID_END}): {len(valid_df)} 条")
    
    if train_df.empty or valid_df.empty:
        print("错误：训练集或验证集为空，请检查时间设置或数据源。")
        exit()

    # 计算基准胜率 (Base Rate)
    base_rate = valid_df[LABEL].mean()
    print(f"   验证集原始胜率 (Base Rate): {base_rate*100:.2f}%")

    # 3. 初始化训练器
    trainer = ModelTrainer(train_df, valid_df)
    results = []

    # ------------------ 模型竞技场 ------------------
    
    # Model 1: LightGBM
    lgb_model, lgb_pred = trainer.run_lightgbm()
    valid_df['score_lgb'] = lgb_pred
    acc_lgb = evaluate_top_k(valid_df, 'LightGBM', pred_col='score_lgb', top_n=5)
    results.append({'Model': 'LightGBM', 'Top5_WinRate': acc_lgb, 'Object': lgb_model})
    
    # Model 2: XGBoost
    xgb_model, xgb_pred = trainer.run_xgboost()
    valid_df['score_xgb'] = xgb_pred
    acc_xgb = evaluate_top_k(valid_df, 'XGBoost', pred_col='score_xgb', top_n=5)
    results.append({'Model': 'XGBoost', 'Top5_WinRate': acc_xgb, 'Object': xgb_model})
    
    # Model 3: CatBoost
    cat_model, cat_pred = trainer.run_catboost()
    valid_df['score_cat'] = cat_pred
    acc_cat = evaluate_top_k(valid_df, 'CatBoost', pred_col='score_cat', top_n=5)
    results.append({'Model': 'CatBoost', 'Top5_WinRate': acc_cat, 'Object': cat_model})

    # Model 4: 平均融合 (Ensemble)
    # 简单的把三个模型的分数加起来取平均，往往最稳
    valid_df['score_avg'] = (valid_df['score_lgb'] + valid_df['score_xgb'] + valid_df['score_cat']) / 3
    acc_avg = evaluate_top_k(valid_df, 'Ensemble(Avg)', pred_col='score_avg', top_n=5)
    results.append({'Model': 'Ensemble', 'Top5_WinRate': acc_avg, 'Object': None})

    # ------------------ 结果展示 ------------------
    print("\n" + "="*40)
    print("       模型竞技结果 (Top 5 命中率)")
    print("="*40)
    
    results_df = pd.DataFrame(results).sort_values(by='Top5_WinRate', ascending=False)
    
    print(f"基准胜率 (瞎买): {base_rate*100:.2f}%")
    print("-" * 40)
    for index, row in results_df.iterrows():
        lift = row['Top5_WinRate'] / base_rate
        print(f"[{row['Model']}] 胜率: {row['Top5_WinRate']*100:.2f}% | 提升度: {lift:.2f}倍")
    print("-" * 40)
    
    # 4. 保存最佳模型
    best_model_info = results_df.iloc[0]
    best_model_name = best_model_info['Model']
    
    if best_model_name == 'Ensemble':
        print("最佳策略是三个模型融合。由于代码演示原因，保存单体最强模型。")
        # 找到单体最强的
        best_single = results_df[results_df['Model'] != 'Ensemble'].iloc[0]
        best_model_obj = best_single['Object']
        best_name = best_single['Model']
    else:
        best_model_obj = best_model_info['Object']
        best_name = best_model_name
        
    save_path = os.path.join(MODEL_DIR, f"best_model_{best_name}.pkl")
    joblib.dump(best_model_obj, save_path)
    # ... (接在之前的代码后面) ...

    # ================= 6. 输出最后一天实战选股结果 =================
    print("\n" + "="*40)
    print("       🔥 最后一天实战选股模拟 🔥")
    print("="*40)
    
    # 1. 获取验证集中最后一天
    last_date = valid_df[DATE_COL].max()
    print(f"日期: {last_date.date()}")
    
    # 2. 提取当天数据
    last_day_df = valid_df[valid_df[DATE_COL] == last_date].copy()
    
    if last_day_df.empty:
        print("错误：未找到最后一天的数据。")
    else:
        # 定义一个帮助函数来展示 Top 5
        def print_top_picks(df, model_name, score_col):
            print(f"\n[{model_name}] 推荐 Top 5:")
            # 按分数降序
            top_5 = df.sort_values(by=score_col, ascending=False).head(10)
            
            # 整理输出格式
            display_cols = ['code', score_col, LABEL] # 如果 label 还是空的(因为是未来)，可能显示 NaN
            
            # 格式化输出
            print(f"{'Code':<10} | {'Score':<10} | {'Label (T+1)'}")
            print("-" * 35)
            for _, row in top_5.iterrows():
                # 转换分数为百分比显示
                score_str = f"{row[score_col]:.4f}"
                label_str = str(int(row[LABEL])) if not pd.isna(row[LABEL]) else "?"
                print(f"{row['code']:<10} | {score_str:<10} | {label_str}")

        # 3. 分别输出各模型的推荐
        # 确保列名和上面训练时保存的一致
        if 'score_cat' in last_day_df.columns:
            print_top_picks(last_day_df, "CatBoost (冠军)", 'score_cat')
            
        if 'score_lgb' in last_day_df.columns:
            print_top_picks(last_day_df, "LightGBM", 'score_lgb')
            
        if 'score_xgb' in last_day_df.columns:
            print_top_picks(last_day_df, "XGBoost", 'score_xgb')
            
        if 'score_avg' in last_day_df.columns:
            print_top_picks(last_day_df, "Ensemble (融合)", 'score_avg')

    print("\n提示：Label=1 表示第二天确实冲高成功，Label=0 表示失败。")
    print("如果这是最新的数据，Label 可能是已知的（因为是回测验证集）。")
    print(f"\n最佳单体模型 [{best_name}] 已保存至: {save_path}")
    
    # 也可以把回测数据存下来看细节
    valid_df.to_csv(os.path.join(MODEL_DIR, "model_comparison_results.csv"), index=False)

1. 读取数据: D:\work\data2\train_data.csv ...
2. 切分数据集...
   训练集 (2020-01-01 ~ 2025-01-01): 87444 条
   验证集 (2025-01-01 ~ 2025-12-31): 17563 条
   验证集原始胜率 (Base Rate): 32.77%

--- Training LightGBM ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[18]	valid_0's auc: 0.584512

--- Training XGBoost ---

--- Training CatBoost ---

       模型竞技结果 (Top 5 命中率)
基准胜率 (瞎买): 32.77%
----------------------------------------
[CatBoost] 胜率: 45.63% | 提升度: 1.39倍
[Ensemble] 胜率: 44.59% | 提升度: 1.36倍
[LightGBM] 胜率: 43.64% | 提升度: 1.33倍
[XGBoost] 胜率: 42.86% | 提升度: 1.31倍
----------------------------------------

       🔥 最后一天实战选股模拟 🔥
日期: 2025-12-15

[CatBoost (冠军)] 推荐 Top 5:
Code       | Score      | Label (T+1)
-----------------------------------
534        | 0.4766     | 0
600990     | 0.4552     | 0
603031     | 0.4490     | 0
605090     | 0.4408     | 0
2157       | 0.4364     | 0
600776     | 0.4300     | 0
605123     | 0.4286     | 0
603992     | 0.4198     |

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ================= 配置区 =================
DATA_PATH = r"D:\work\data2\train_data_T+2.csv"
MODEL_DIR = r"D:\work\data"  # 模型保存目录

# --- 1. 时间窗口设置 (请根据你的数据实际时间修改) ---
# 格式: 'YYYY-MM-DD'
TRAIN_START = '2020-01-01'
TRAIN_END   = '2025-01-01'

VALID_START = '2025-01-01'
VALID_END   = '2025-12-31'

# --- 2. 特征与标签 ---
FEATURES = [
    'upper_shadow',      # 上影线
    'ma5_bias',          # 均线乖离
    'vol_ratio',         # 量比
    'limit_up_count',    # 大盘涨停数
    'avg_pct_change',    # 大盘平均涨跌幅
    'body_size',         # 实体大小
    'high_low_ratio'     # 振幅
]
LABEL = 'label'
DATE_COL = 'date'

# ================= 核心工具函数 =================

def load_data(path):
    print(f"1. 读取数据: {path} ...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    
    # 简单的缺失值填充 (树模型其实可以不填，但逻辑回归需要)
    df = df.dropna(subset=[LABEL]) # 标签为空必须删
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].mean())
    
    return df.sort_values(DATE_COL)

def get_dataset_by_date(df, start_date, end_date):
    """根据时间段截取数据"""
    mask = (df[DATE_COL] >= start_date) & (df[DATE_COL] <= end_date)
    return df[mask].copy()

def evaluate_top_k(df, model_name, pred_col='pred_score', top_n=5):
    """计算 Top 5 实战命中率"""
    daily_precisions = []
    
    for date, group in df.groupby(DATE_COL):
        if len(group) < 1: continue
        
        # 选分最高的 Top N
        top_picks = group.sort_values(by=pred_col, ascending=False).head(top_n)
        
        # 计算命中数
        hits = top_picks[LABEL].sum()
        # 实际选了多少只 (有可能那天只有3只票)
        actual_k = len(top_picks)
        
        if actual_k > 0:
            daily_precisions.append(hits / actual_k)
            
    if not daily_precisions:
        return 0.0
        
    return np.mean(daily_precisions)

# ================= 模型训练包装类 =================

class ModelTrainer:
    def __init__(self, train_df, valid_df):
        self.train_df = train_df
        self.valid_df = valid_df

        self.X_train = train_df[FEATURES]
        self.y_train = train_df[LABEL]
        self.w_train = train_df['weight']

        self.X_valid = valid_df[FEATURES]
        self.y_valid = valid_df[LABEL]
        self.w_valid = valid_df['weight']

        self.valid_df_raw = valid_df.copy()


    def run_lightgbm(self):
        print("\n--- Training LightGBM ---")
        train_data = lgb.Dataset(
            self.X_train,
            label=self.y_train,
            weight=self.w_train
        )

        valid_data = lgb.Dataset(self.X_valid, label=self.y_valid, reference=train_data)
        
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'learning_rate': 0.05,
            'verbose': -1,
            'n_jobs': -1,
            'seed': 42
        }
        
        model = lgb.train(
            params, train_data, num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)] # 0表示不刷屏
        )
        return model, model.predict(self.X_valid)

    def run_xgboost(self):
            print("\n--- Training XGBoost ---")
            # 修改点 1: 将 early_stopping_rounds 放入初始化函数中
            model = xgb.XGBClassifier(
                objective='binary:logistic',
                eval_metric='auc',
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=5,
                n_jobs=-1,
                random_state=42,
                early_stopping_rounds=50  # <--- 从 fit() 移到这里
            )
            
            # 修改点 2: fit() 中去掉 early_stopping_rounds
            model.fit(
                self.X_train, self.y_train,
                eval_set=[(self.X_valid, self.y_valid)],
                verbose=False
                # early_stopping_rounds=50  <--- 这里删掉
            )
            # XGBoost predict_proba 返回 [class0_prob, class1_prob]
            return model, model.predict_proba(self.X_valid)[:, 1]

    def run_catboost(self):
        print("\n--- Training CatBoost ---")
        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=42,
            allow_writing_files=False
        )
        
        model.fit(
            self.X_train, self.y_train,
            eval_set=(self.X_valid, self.y_valid),
            early_stopping_rounds=50
        )
        return model, model.predict_proba(self.X_valid)[:, 1]

# ================= 主流程 =================

if __name__ == "__main__":
    # 1. 加载全量数据
    try:
        full_df = load_data(DATA_PATH)
    except Exception as e:
        print(e)
        exit()

    # 2. 按指定时间切分
    print(f"2. 切分数据集...")
    train_df = get_dataset_by_date(full_df, TRAIN_START, TRAIN_END)
    valid_df = get_dataset_by_date(full_df, VALID_START, VALID_END)
    # ================= 样本权重（只影响训练，不影响验证） =================

    # 基础权重
    train_df['weight'] = 1.0

    # 正样本更重要（你可以从 2.0 / 3.0 开始试）
    train_df.loc[train_df[LABEL] == 1, 'weight'] = 3.0

    # 验证集权重不重要，但为了接口统一给一个
    valid_df['weight'] = 1.0

    print(f"   训练集 ({TRAIN_START} ~ {TRAIN_END}): {len(train_df)} 条")
    print(f"   验证集 ({VALID_START} ~ {VALID_END}): {len(valid_df)} 条")
    
    if train_df.empty or valid_df.empty:
        print("错误：训练集或验证集为空，请检查时间设置或数据源。")
        exit()

    # 计算基准胜率 (Base Rate)
    base_rate = valid_df[LABEL].mean()
    print(f"   验证集原始胜率 (Base Rate): {base_rate*100:.2f}%")

    # 3. 初始化训练器
    trainer = ModelTrainer(train_df, valid_df)
    results = []

    # ------------------ 模型竞技场 ------------------
    
    # Model 1: LightGBM
    lgb_model, lgb_pred = trainer.run_lightgbm()
    valid_df['score_lgb'] = lgb_pred
    acc_lgb = evaluate_top_k(valid_df, 'LightGBM', pred_col='score_lgb', top_n=5)
    results.append({'Model': 'LightGBM', 'Top5_WinRate': acc_lgb, 'Object': lgb_model})
    
    # Model 2: XGBoost
    xgb_model, xgb_pred = trainer.run_xgboost()
    valid_df['score_xgb'] = xgb_pred
    acc_xgb = evaluate_top_k(valid_df, 'XGBoost', pred_col='score_xgb', top_n=5)
    results.append({'Model': 'XGBoost', 'Top5_WinRate': acc_xgb, 'Object': xgb_model})
    
    # Model 3: CatBoost
    cat_model, cat_pred = trainer.run_catboost()
    valid_df['score_cat'] = cat_pred
    acc_cat = evaluate_top_k(valid_df, 'CatBoost', pred_col='score_cat', top_n=5)
    results.append({'Model': 'CatBoost', 'Top5_WinRate': acc_cat, 'Object': cat_model})

    # Model 4: 平均融合 (Ensemble)
    # 简单的把三个模型的分数加起来取平均，往往最稳
    valid_df['score_avg'] = (valid_df['score_lgb'] + valid_df['score_xgb'] + valid_df['score_cat']) / 3
    acc_avg = evaluate_top_k(valid_df, 'Ensemble(Avg)', pred_col='score_avg', top_n=5)
    results.append({'Model': 'Ensemble', 'Top5_WinRate': acc_avg, 'Object': None})

    # ------------------ 结果展示 ------------------
    print("\n" + "="*40)
    print("       模型竞技结果 (Top 5 命中率)")
    print("="*40)
    
    results_df = pd.DataFrame(results).sort_values(by='Top5_WinRate', ascending=False)
    
    print(f"基准胜率 (瞎买): {base_rate*100:.2f}%")
    print("-" * 40)
    for index, row in results_df.iterrows():
        lift = row['Top5_WinRate'] / base_rate
        print(f"[{row['Model']}] 胜率: {row['Top5_WinRate']*100:.2f}% | 提升度: {lift:.2f}倍")
    print("-" * 40)
    
    # 4. 保存最佳模型
    best_model_info = results_df.iloc[0]
    best_model_name = best_model_info['Model']
    
    if best_model_name == 'Ensemble':
        print("最佳策略是三个模型融合。由于代码演示原因，保存单体最强模型。")
        # 找到单体最强的
        best_single = results_df[results_df['Model'] != 'Ensemble'].iloc[0]
        best_model_obj = best_single['Object']
        best_name = best_single['Model']
    else:
        best_model_obj = best_model_info['Object']
        best_name = best_model_name
        
    save_path = os.path.join(MODEL_DIR, f"best_model_{best_name}.pkl")
    joblib.dump(best_model_obj, save_path)
    # ... (接在之前的代码后面) ...

    # ================= 6. 输出最后一天实战选股结果 =================
    print("\n" + "="*40)
    print("       🔥 最后一天实战选股模拟 🔥")
    print("="*40)
    
    # 1. 获取验证集中最后一天
    last_date = valid_df[DATE_COL].max()
    print(f"日期: {last_date.date()}")
    
    # 2. 提取当天数据
    last_day_df = valid_df[valid_df[DATE_COL] == last_date].copy()
    
    if last_day_df.empty:
        print("错误：未找到最后一天的数据。")
    else:
        # 定义一个帮助函数来展示 Top 5
        def print_top_picks(df, model_name, score_col):
            print(f"\n[{model_name}] 推荐 Top 5:")
            # 按分数降序
            top_5 = df.sort_values(by=score_col, ascending=False).head(10)
            
            # 整理输出格式
            display_cols = ['code', score_col, LABEL] # 如果 label 还是空的(因为是未来)，可能显示 NaN
            
            # 格式化输出
            print(f"{'Code':<10} | {'Score':<10} | {'Label (T+1)'}")
            print("-" * 35)
            for _, row in top_5.iterrows():
                # 转换分数为百分比显示
                score_str = f"{row[score_col]:.4f}"
                label_str = str(int(row[LABEL])) if not pd.isna(row[LABEL]) else "?"
                print(f"{row['code']:<10} | {score_str:<10} | {label_str}")

        # 3. 分别输出各模型的推荐
        # 确保列名和上面训练时保存的一致
        if 'score_cat' in last_day_df.columns:
            print_top_picks(last_day_df, "CatBoost (冠军)", 'score_cat')
            
        if 'score_lgb' in last_day_df.columns:
            print_top_picks(last_day_df, "LightGBM", 'score_lgb')
            
        if 'score_xgb' in last_day_df.columns:
            print_top_picks(last_day_df, "XGBoost", 'score_xgb')
            
        if 'score_avg' in last_day_df.columns:
            print_top_picks(last_day_df, "Ensemble (融合)", 'score_avg')

    print("\n提示：Label=1 表示第二天确实冲高成功，Label=0 表示失败。")
    print("如果这是最新的数据，Label 可能是已知的（因为是回测验证集）。")
    print(f"\n最佳单体模型 [{best_name}] 已保存至: {save_path}")
    
    # 也可以把回测数据存下来看细节
    valid_df.to_csv(os.path.join(MODEL_DIR, "model_comparison_results.csv"), index=False)

1. 读取数据: D:\work\data2\train_data_T+2.csv ...
2. 切分数据集...
   训练集 (2020-01-01 ~ 2025-01-01): 87497 条
   验证集 (2025-01-01 ~ 2025-12-31): 17767 条
   验证集原始胜率 (Base Rate): 27.19%

--- Training LightGBM ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[52]	valid_0's auc: 0.543065

--- Training XGBoost ---

--- Training CatBoost ---

       模型竞技结果 (Top 5 命中率)
基准胜率 (瞎买): 27.19%
----------------------------------------
[Ensemble] 胜率: 33.16% | 提升度: 1.22倍
[CatBoost] 胜率: 32.14% | 提升度: 1.18倍
[LightGBM] 胜率: 31.88% | 提升度: 1.17倍
[XGBoost] 胜率: 30.00% | 提升度: 1.10倍
----------------------------------------
最佳策略是三个模型融合。由于代码演示原因，保存单体最强模型。

       🔥 最后一天实战选股模拟 🔥
日期: 2025-12-18

[CatBoost (冠军)] 推荐 Top 5:
Code       | Score      | Label (T+1)
-----------------------------------
2198       | 0.3320     | 0
2491       | 0.3313     | 0
2837       | 0.3285     | 0
603863     | 0.3268     | 0
603708     | 0.3252     | 0
603090     | 0.3168     | 0
600879     | 0.314

In [27]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ================= 配置区 =================
DATA_PATH = r"D:\work\data2\train_data.csv"
MODEL_DIR = r"D:\work\data"  # 模型保存目录

# --- 1. 时间窗口设置 (请根据你的数据实际时间修改) ---
# 格式: 'YYYY-MM-DD'
TRAIN_START = '2020-01-01'
TRAIN_END   = '2025-01-01'

VALID_START = '2025-01-01'
VALID_END   = '2025-12-31'

# --- 2. 特征与标签 ---
FEATURES = [
    'upper_shadow',      # 上影线
    'ma5_bias',          # 均线乖离
    'vol_ratio',         # 量比
    'limit_up_count',    # 大盘涨停数
    'avg_pct_change',    # 大盘平均涨跌幅
    'body_size',         # 实体大小
    'high_low_ratio'     # 振幅
]
LABEL = 'label'
DATE_COL = 'date'

# --- 3. 新增对比组参数 ---
TOP_K = 8  # 可调的K值
THRESHOLD = -0.02 # next_open_change阈值，单位：百分比
NEXT_OPEN_CHANGE_COL = 'next_open_change'  # 新增的列名

# ================= 核心工具函数 =================

def load_data(path):
    print(f"1. 读取数据: {path} ...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    
    # 检查是否有next_open_change列
    if NEXT_OPEN_CHANGE_COL not in df.columns:
        print(f"警告: 数据中没有找到 {NEXT_OPEN_CHANGE_COL} 列！")
        # 如果没有该列，创建一个虚拟列（实际使用时请确保数据中有该列）
        df[NEXT_OPEN_CHANGE_COL] = np.random.uniform(-5, 5, len(df))
    
    # 简单的缺失值填充
    df = df.dropna(subset=[LABEL]) # 标签为空必须删
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].mean())
    
    # 填充next_open_change的缺失值
    if NEXT_OPEN_CHANGE_COL in df.columns:
        df[NEXT_OPEN_CHANGE_COL] = df[NEXT_OPEN_CHANGE_COL].fillna(df[NEXT_OPEN_CHANGE_COL].mean())
    
    return df.sort_values(DATE_COL)

def get_dataset_by_date(df, start_date, end_date):
    """根据时间段截取数据"""
    mask = (df[DATE_COL] >= start_date) & (df[DATE_COL] <= end_date)
    return df[mask].copy()

def evaluate_top_k(df, model_name, pred_col='pred_score', top_n=TOP_K, 
                   filter_col=None, threshold=None):
    """
    计算Top K实战命中率，支持筛选条件
    
    Parameters:
    -----------
    df : DataFrame
        包含预测结果的数据
    model_name : str
        模型名称
    pred_col : str
        预测分数列名
    top_n : int
        选取的Top K数量
    filter_col : str or None
        筛选列名，如'next_open_change'
    threshold : float or None
        筛选阈值，如-2.0表示筛选大于-2%的
    
    Returns:
    --------
    dict : 包含胜率、命中数、总票数等信息
    """
    daily_results = []
    
    for date, group in df.groupby(DATE_COL):
        if len(group) < 1: 
            continue
        
        # 按预测分数排序
        sorted_group = group.sort_values(by=pred_col, ascending=False)
        
        # 原始Top K
        top_picks = sorted_group.head(top_n)
        
        # 应用筛选条件（如果提供了）
        if filter_col is not None and threshold is not None:
            filtered_top_picks = top_picks[top_picks[filter_col] > threshold]
        else:
            filtered_top_picks = top_picks.copy()
        
        # 如果没有符合条件的股票，跳过这一天
        if len(filtered_top_picks) == 0:
            continue
        
        # 计算命中数
        hits = filtered_top_picks[LABEL].sum()
        total_picks = len(filtered_top_picks)
        
        daily_results.append({
            'date': date,
            'hits': hits,
            'total': total_picks,
            'win_rate': hits / total_picks if total_picks > 0 else 0
        })
    
    if not daily_results:
        return {
            'win_rate': 0.0,
            'hits': 0,
            'total_picks': 0,
            'days_with_picks': 0
        }
    
    # 汇总所有日期的结果
    results_df = pd.DataFrame(daily_results)
    
    return {
        'win_rate': results_df['win_rate'].mean(),
        'hits': results_df['hits'].sum(),
        'total_picks': results_df['total'].sum(),
        'days_with_picks': len(results_df),
        'avg_picks_per_day': results_df['total'].mean()
    }

# ================= 模型训练包装类 =================

class ModelTrainer:
    def __init__(self, train_df, valid_df):
        self.train_df = train_df
        self.valid_df = valid_df

        self.X_train = train_df[FEATURES]
        self.y_train = train_df[LABEL]
        self.w_train = train_df['weight']

        self.X_valid = valid_df[FEATURES]
        self.y_valid = valid_df[LABEL]
        self.w_valid = valid_df['weight']

        self.valid_df_raw = valid_df.copy()

    def run_lightgbm(self):
        print("\n--- Training LightGBM ---")
        train_data = lgb.Dataset(
            self.X_train,
            label=self.y_train,
            weight=self.w_train
        )

        valid_data = lgb.Dataset(self.X_valid, label=self.y_valid, reference=train_data)
        
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'learning_rate': 0.05,
            'verbose': -1,
            'n_jobs': -1,
            'seed': 42
        }
        
        model = lgb.train(
            params, train_data, num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
        return model, model.predict(self.X_valid)

    def run_xgboost(self):
        print("\n--- Training XGBoost ---")
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=5,
            n_jobs=-1,
            random_state=42,
            early_stopping_rounds=50
        )
        
        model.fit(
            self.X_train, self.y_train,
            eval_set=[(self.X_valid, self.y_valid)],
            verbose=False
        )
        return model, model.predict_proba(self.X_valid)[:, 1]

    def run_catboost(self):
        print("\n--- Training CatBoost ---")
        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=42,
            allow_writing_files=False
        )
        
        model.fit(
            self.X_train, self.y_train,
            eval_set=(self.X_valid, self.y_valid),
            early_stopping_rounds=50
        )
        return model, model.predict_proba(self.X_valid)[:, 1]

# ================= 主流程 =================

if __name__ == "__main__":
    # 打印参数设置
    print(f"对比组参数设置:")
    print(f"  TOP_K = {TOP_K}")
    print(f"  THRESHOLD = {THRESHOLD}%")
    print(f"  筛选条件: {NEXT_OPEN_CHANGE_COL} > {THRESHOLD}%\n")
    
    # 1. 加载全量数据
    try:
        full_df = load_data(DATA_PATH)
    except Exception as e:
        print(e)
        exit()

    # 2. 按指定时间切分
    print(f"2. 切分数据集...")
    train_df = get_dataset_by_date(full_df, TRAIN_START, TRAIN_END)
    valid_df = get_dataset_by_date(full_df, VALID_START, VALID_END)
    
    # 设置样本权重
    train_df['weight'] = 1.0
    train_df.loc[train_df[LABEL] == 1, 'weight'] = 3.0
    valid_df['weight'] = 1.0

    print(f"   训练集 ({TRAIN_START} ~ {TRAIN_END}): {len(train_df)} 条")
    print(f"   验证集 ({VALID_START} ~ {VALID_END}): {len(valid_df)} 条")
    
    if train_df.empty or valid_df.empty:
        print("错误：训练集或验证集为空，请检查时间设置或数据源。")
        exit()

    # 计算基准胜率 (Base Rate)
    base_rate = valid_df[LABEL].mean()
    print(f"   验证集原始胜率 (Base Rate): {base_rate*100:.2f}%")

    # 3. 初始化训练器
    trainer = ModelTrainer(train_df, valid_df)
    
    # 用于存储所有对比组的结果
    comparison_results = []

    # ------------------ 模型竞技场 ------------------
    
    # Model 1: LightGBM
    print("\n" + "="*60)
    print("LightGBM 对比测试")
    print("="*60)
    lgb_model, lgb_pred = trainer.run_lightgbm()
    valid_df['score_lgb'] = lgb_pred
    
    # 原始Top K
    lgb_original = evaluate_top_k(valid_df, 'LightGBM', pred_col='score_lgb')
    
    # 筛选后的Top K
    lgb_filtered = evaluate_top_k(valid_df, 'LightGBM', pred_col='score_lgb',
                                  filter_col=NEXT_OPEN_CHANGE_COL, threshold=THRESHOLD)
    
    comparison_results.append({
        'Model': 'LightGBM',
        'Type': '原始TopK',
        'TopK': TOP_K,
        'WinRate': lgb_original['win_rate'],
        'Hits': lgb_original['hits'],
        'TotalPicks': lgb_original['total_picks'],
        'DaysWithPicks': lgb_original['days_with_picks'],
        'Object': lgb_model
    })
    
    comparison_results.append({
        'Model': 'LightGBM',
        'Type': f'筛选后TopK\n({NEXT_OPEN_CHANGE_COL}>{THRESHOLD}%)',
        'TopK': TOP_K,
        'WinRate': lgb_filtered['win_rate'],
        'Hits': lgb_filtered['hits'],
        'TotalPicks': lgb_filtered['total_picks'],
        'DaysWithPicks': lgb_filtered['days_with_picks'],
        'Object': None
    })
    
    # Model 2: XGBoost
    print("\n" + "="*60)
    print("XGBoost 对比测试")
    print("="*60)
    xgb_model, xgb_pred = trainer.run_xgboost()
    valid_df['score_xgb'] = xgb_pred
    
    xgb_original = evaluate_top_k(valid_df, 'XGBoost', pred_col='score_xgb')
    xgb_filtered = evaluate_top_k(valid_df, 'XGBoost', pred_col='score_xgb',
                                  filter_col=NEXT_OPEN_CHANGE_COL, threshold=THRESHOLD)
    
    comparison_results.append({
        'Model': 'XGBoost',
        'Type': '原始TopK',
        'TopK': TOP_K,
        'WinRate': xgb_original['win_rate'],
        'Hits': xgb_original['hits'],
        'TotalPicks': xgb_original['total_picks'],
        'DaysWithPicks': xgb_original['days_with_picks'],
        'Object': xgb_model
    })
    
    comparison_results.append({
        'Model': 'XGBoost',
        'Type': f'筛选后TopK\n({NEXT_OPEN_CHANGE_COL}>{THRESHOLD}%)',
        'TopK': TOP_K,
        'WinRate': xgb_filtered['win_rate'],
        'Hits': xgb_filtered['hits'],
        'TotalPicks': xgb_filtered['total_picks'],
        'DaysWithPicks': xgb_filtered['days_with_picks'],
        'Object': None
    })
    
    # Model 3: CatBoost
    print("\n" + "="*60)
    print("CatBoost 对比测试")
    print("="*60)
    cat_model, cat_pred = trainer.run_catboost()
    valid_df['score_cat'] = cat_pred
    
    cat_original = evaluate_top_k(valid_df, 'CatBoost', pred_col='score_cat')
    cat_filtered = evaluate_top_k(valid_df, 'CatBoost', pred_col='score_cat',
                                  filter_col=NEXT_OPEN_CHANGE_COL, threshold=THRESHOLD)
    
    comparison_results.append({
        'Model': 'CatBoost',
        'Type': '原始TopK',
        'TopK': TOP_K,
        'WinRate': cat_original['win_rate'],
        'Hits': cat_original['hits'],
        'TotalPicks': cat_original['total_picks'],
        'DaysWithPicks': cat_original['days_with_picks'],
        'Object': cat_model
    })
    
    comparison_results.append({
        'Model': 'CatBoost',
        'Type': f'筛选后TopK\n({NEXT_OPEN_CHANGE_COL}>{THRESHOLD}%)',
        'TopK': TOP_K,
        'WinRate': cat_filtered['win_rate'],
        'Hits': cat_filtered['hits'],
        'TotalPicks': cat_filtered['total_picks'],
        'DaysWithPicks': cat_filtered['days_with_picks'],
        'Object': None
    })
    
    # Model 4: 平均融合 (Ensemble)
    print("\n" + "="*60)
    print("Ensemble 对比测试")
    print("="*60)
    valid_df['score_avg'] = (valid_df['score_lgb'] + valid_df['score_xgb'] + valid_df['score_cat']) / 3
    
    ensemble_original = evaluate_top_k(valid_df, 'Ensemble', pred_col='score_avg')
    ensemble_filtered = evaluate_top_k(valid_df, 'Ensemble', pred_col='score_avg',
                                       filter_col=NEXT_OPEN_CHANGE_COL, threshold=THRESHOLD)
    
    comparison_results.append({
        'Model': 'Ensemble',
        'Type': '原始TopK',
        'TopK': TOP_K,
        'WinRate': ensemble_original['win_rate'],
        'Hits': ensemble_original['hits'],
        'TotalPicks': ensemble_original['total_picks'],
        'DaysWithPicks': ensemble_original['days_with_picks'],
        'Object': None
    })
    
    comparison_results.append({
        'Model': 'Ensemble',
        'Type': f'筛选后TopK\n({NEXT_OPEN_CHANGE_COL}>{THRESHOLD}%)',
        'TopK': TOP_K,
        'WinRate': ensemble_filtered['win_rate'],
        'Hits': ensemble_filtered['hits'],
        'TotalPicks': ensemble_filtered['total_picks'],
        'DaysWithPicks': ensemble_filtered['days_with_picks'],
        'Object': None
    })

    # ------------------ 结果展示 ------------------
    print("\n" + "="*80)
    print(f"                   模型对比结果 (Top {TOP_K} 命中率)")
    print(f"             筛选条件: {NEXT_OPEN_CHANGE_COL} > {THRESHOLD}%")
    print("="*80)
    
    comp_df = pd.DataFrame(comparison_results)
    
    print(f"\n基准胜率 (全量数据): {base_rate*100:.2f}%")
    print(f"总交易天数: {valid_df[DATE_COL].nunique()}")
    print("-" * 80)
    
    # 按模型分组展示
    models = comp_df['Model'].unique()
    
    for model in models:
        model_results = comp_df[comp_df['Model'] == model]
        print(f"\n📊 {model}:")
        print("-" * 40)
        
        for _, row in model_results.iterrows():
            lift = row['WinRate'] / base_rate if base_rate > 0 else 0
            avg_picks = row['TotalPicks'] / row['DaysWithPicks'] if row['DaysWithPicks'] > 0 else 0
            
            print(f"  {row['Type']}:")
            print(f"    胜率: {row['WinRate']*100:.2f}% | 提升度: {lift:.2f}倍")
            print(f"    命中数: {row['Hits']}/{row['TotalPicks']} | 有选股天数: {row['DaysWithPicks']}")
            print(f"    日均选股数: {avg_picks:.1f}")
    
    print("\n" + "="*80)
    print("                        综合排名")
    print("="*80)
    
    # 按胜率排序
    ranked_df = comp_df.sort_values(by='WinRate', ascending=False).reset_index(drop=True)
    
    print(f"\n{'排名':<5} {'模型':<12} {'策略类型':<25} {'胜率':<10} {'提升度':<10} {'命中数':<15}")
    print("-" * 80)
    
    for idx, row in ranked_df.iterrows():
        lift = row['WinRate'] / base_rate if base_rate > 0 else 0
        hit_rate = f"{row['Hits']}/{row['TotalPicks']}"
        
        print(f"{idx+1:<5} {row['Model']:<12} {row['Type']:<25} "
              f"{row['WinRate']*100:>6.2f}% {lift:>8.2f}x {hit_rate:>15}")

    # 4. 保存最佳模型（考虑筛选前后的最佳策略）
    # 找出原始TopK策略中最佳模型
    original_results = comp_df[comp_df['Type'] == '原始TopK']
    best_original = original_results.loc[original_results['WinRate'].idxmax()]
    
    # 找出筛选后TopK策略中最佳模型
    filtered_type = f'筛选后TopK\n({NEXT_OPEN_CHANGE_COL}>{THRESHOLD}%)'
    filtered_results = comp_df[comp_df['Type'] == filtered_type]
    
    if len(filtered_results) > 0:
        best_filtered = filtered_results.loc[filtered_results['WinRate'].idxmax()]
        
        print(f"\n🎯 最佳原始策略: {best_original['Model']} (胜率: {best_original['WinRate']*100:.2f}%)")
        print(f"🎯 最佳筛选策略: {best_filtered['Model']} (胜率: {best_filtered['WinRate']*100:.2f}%)")
        
        # 保存两个策略的模型
        if best_original['Object'] is not None:
            save_path_original = os.path.join(MODEL_DIR, f"best_original_{best_original['Model']}_K{TOP_K}.pkl")
            joblib.dump(best_original['Object'], save_path_original)
            print(f"💾 原始策略模型已保存至: {save_path_original}")
    
    # ================= 最后一天实战选股结果 =================
    print("\n" + "="*80)
    print("                      🔥 最后一天实战选股模拟 🔥")
    print("="*80)
    
    last_date = valid_df[DATE_COL].max()
    print(f"📅 日期: {last_date.date()}")
    print(f"⚙️  参数: Top{TOP_K}, 筛选条件: {NEXT_OPEN_CHANGE_COL} > {THRESHOLD}%\n")
    
    last_day_df = valid_df[valid_df[DATE_COL] == last_date].copy()
    
    if last_day_df.empty:
        print("错误：未找到最后一天的数据。")
    else:
        # 定义一个帮助函数来展示选股结果
        def print_picks(df, model_name, score_col, show_filtered=True):
            print(f"\n[{model_name}] 推荐结果:")
            print("-" * 70)
            
            # 原始Top K
            top_k = df.sort_values(by=score_col, ascending=False).head(TOP_K)
            print(f"原始Top {TOP_K}:")
            print(f"{'代码':<10} | {'分数':<10} | {'次日涨跌':<10} | {NEXT_OPEN_CHANGE_COL}")
            print("-" * 70)
            
            for _, row in top_k.iterrows():
                next_open = row[NEXT_OPEN_CHANGE_COL] if NEXT_OPEN_CHANGE_COL in row else 'N/A'
                meets_criteria = next_open != 'N/A' and next_open > THRESHOLD
                flag = "✅" if meets_criteria else "❌"
                
                print(f"{row['code']:<10} | {row[score_col]:<10.4f} | "
                      f"{row[LABEL]:<10} | {next_open:<10.2f} {flag}")
            
            # 筛选后的结果
            if show_filtered and NEXT_OPEN_CHANGE_COL in df.columns:
                filtered = top_k[top_k[NEXT_OPEN_CHANGE_COL] > THRESHOLD]
                print(f"\n筛选后 ({NEXT_OPEN_CHANGE_COL} > {THRESHOLD}%):")
                print(f"符合条件的股票数: {len(filtered)}/{TOP_K}")
                
                if len(filtered) > 0:
                    print(f"{'代码':<10} | {'分数':<10} | {'次日涨跌':<10} | {NEXT_OPEN_CHANGE_COL}")
                    print("-" * 70)
                    for _, row in filtered.iterrows():
                        print(f"{row['code']:<10} | {row[score_col]:<10.4f} | "
                              f"{row[LABEL]:<10} | {row[NEXT_OPEN_CHANGE_COL]:<10.2f}")
        
        # 分别输出各模型的推荐
        models_to_check = [
            ('CatBoost', 'score_cat'),
            ('LightGBM', 'score_lgb'),
            ('XGBoost', 'score_xgb'),
            ('Ensemble', 'score_avg')
        ]
        
        for model_name, score_col in models_to_check:
            if score_col in last_day_df.columns:
                print_picks(last_day_df, model_name, score_col)
    
    # 保存详细的对比结果
    comp_df.to_csv(os.path.join(MODEL_DIR, f"model_comparison_K{TOP_K}_threshold{THRESHOLD}.csv"), index=False)
    valid_df.to_csv(os.path.join(MODEL_DIR, "detailed_predictions.csv"), index=False)
    
    print(f"\n📊 详细对比结果已保存至: {MODEL_DIR}")

对比组参数设置:
  TOP_K = 8
  THRESHOLD = -0.02%
  筛选条件: next_open_change > -0.02%

1. 读取数据: D:\work\data2\train_data.csv ...
2. 切分数据集...
   训练集 (2020-01-01 ~ 2025-01-01): 87497 条
   验证集 (2025-01-01 ~ 2025-12-31): 17634 条
   验证集原始胜率 (Base Rate): 32.72%

LightGBM 对比测试

--- Training LightGBM ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[13]	valid_0's auc: 0.584973

XGBoost 对比测试

--- Training XGBoost ---

CatBoost 对比测试

--- Training CatBoost ---

Ensemble 对比测试

                   模型对比结果 (Top 8 命中率)
             筛选条件: next_open_change > -0.02%

基准胜率 (全量数据): 32.72%
总交易天数: 232
--------------------------------------------------------------------------------

📊 LightGBM:
----------------------------------------
  原始TopK:
    胜率: 41.92% | 提升度: 1.28倍
    命中数: 778/1856 | 有选股天数: 232
    日均选股数: 8.0
  筛选后TopK
(next_open_change>-0.02%):
    胜率: 39.41% | 提升度: 1.20倍
    命中数: 473/1198 | 有选股天数: 231
    日均选股数: 5.2

📊 XGBoost:
--------------------------------

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ================= 配置区 =================
DATA_PATH = r"D:\work\data2\train_data.csv"
MODEL_DIR = r"D:\work\data"  # 模型保存目录

# --- 1. 时间窗口设置 (请根据你的数据实际时间修改) ---
# 格式: 'YYYY-MM-DD'
TRAIN_START = '2020-01-01'
TRAIN_END   = '2025-01-01'

VALID_START = '2025-01-01'
VALID_END   = '2025-12-31'

# --- 2. 特征与标签 ---
FEATURES = [
    'upper_shadow',      # 上影线
    'ma5_bias',          # 均线乖离
    'vol_ratio',         # 量比
    'limit_up_count',    # 大盘涨停数
    'avg_pct_change',    # 大盘平均涨跌幅
    'body_size',         # 实体大小
    'high_low_ratio'     # 振幅
]
LABEL = 'label'
DATE_COL = 'date'

# ================= 核心工具函数 =================

def load_data(path):
    print(f"1. 读取数据: {path} ...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    
    # 简单的缺失值填充 (树模型其实可以不填，但逻辑回归需要)
    df = df.dropna(subset=[LABEL]) # 标签为空必须删
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].mean())
    
    return df.sort_values(DATE_COL)

def get_dataset_by_date(df, start_date, end_date):
    """根据时间段截取数据"""
    mask = (df[DATE_COL] >= start_date) & (df[DATE_COL] <= end_date)
    return df[mask].copy()

def evaluate_top_k(df, model_name, pred_col='pred_score', top_n=5):
    """计算 Top 5 实战命中率"""
    daily_precisions = []
    
    for date, group in df.groupby(DATE_COL):
        if len(group) < 1: continue
        
        # 选分最高的 Top N
        top_picks = group.sort_values(by=pred_col, ascending=False).head(top_n)
        
        # 计算命中数
        hits = top_picks[LABEL].sum()
        # 实际选了多少只 (有可能那天只有3只票)
        actual_k = len(top_picks)
        
        if actual_k > 0:
            daily_precisions.append(hits / actual_k)
            
    if not daily_precisions:
        return 0.0
        
    return np.mean(daily_precisions)

# ================= 模型训练包装类 =================

class ModelTrainer:
    def __init__(self, train_df, valid_df):
        self.train_df = train_df
        self.valid_df = valid_df

        self.X_train = train_df[FEATURES]
        self.y_train = train_df[LABEL]
        self.w_train = train_df['weight']

        self.X_valid = valid_df[FEATURES]
        self.y_valid = valid_df[LABEL]
        self.w_valid = valid_df['weight']

        self.valid_df_raw = valid_df.copy()


    def run_lightgbm(self):
        print("\n--- Training LightGBM ---")
        train_data = lgb.Dataset(
            self.X_train,
            label=self.y_train,
            weight=self.w_train
        )

        valid_data = lgb.Dataset(self.X_valid, label=self.y_valid, reference=train_data)
         
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'learning_rate': 0.05,
            'verbose': -1,
            'n_jobs': -1,
            'seed': 42
        }
        
        model = lgb.train(
            params, train_data, num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)] # 0表示不刷屏
        )
        return model, model.predict(self.X_valid)

    def run_xgboost(self):
            print("\n--- Training XGBoost ---")
            # 修改点 1: 将 early_stopping_rounds 放入初始化函数中
            model = xgb.XGBClassifier(
                objective='binary:logistic',
                eval_metric='auc',
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=5,
                n_jobs=-1,
                random_state=42,
                early_stopping_rounds=50  # <--- 从 fit() 移到这里
            )
            
            # 修改点 2: fit() 中去掉 early_stopping_rounds
            model.fit(
                self.X_train, self.y_train,
                eval_set=[(self.X_valid, self.y_valid)],
                verbose=False
                # early_stopping_rounds=50  <--- 这里删掉
            )
            # XGBoost predict_proba 返回 [class0_prob, class1_prob]
            return model, model.predict_proba(self.X_valid)[:, 1]

    def run_catboost(self):
        print("\n--- Training CatBoost ---")
        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=42,
            allow_writing_files=False
        )
        
        model.fit(
            self.X_train, self.y_train,
            eval_set=(self.X_valid, self.y_valid),
            early_stopping_rounds=50
        )
        return model, model.predict_proba(self.X_valid)[:, 1]
    def run_lightgbm_rank(self):
        print("\n--- Training LightGBM (LambdaRank for Top-K) ---")

        # ====== 关键：按 date 构造 group ======
        group_train = self.train_df.groupby(DATE_COL).size().to_list()
        group_valid = self.valid_df.groupby(DATE_COL).size().to_list()

        train_data = lgb.Dataset(
            self.X_train,
            label=self.y_train,
            group=group_train,
            weight=self.w_train
        )

        valid_data = lgb.Dataset(
            self.X_valid,
            label=self.y_valid,
            group=group_valid,
            reference=train_data
        )

        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'ndcg_eval_at': [5, 8, 10],   # 和你的 Top-K 对齐
            'learning_rate': 0.05,
            'num_leaves': 31,
            'verbose': -1,
            'seed': 42
        }

        model = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
        )

        # lambdarank 的 predict 依然是 score（用于排序）
        return model, model.predict(self.X_valid)


# ================= 主流程 =================

if __name__ == "__main__":
    # 1. 加载全量数据
    try:
        full_df = load_data(DATA_PATH)
    except Exception as e:
        print(e)
        exit()

    # 2. 按指定时间切分
    print(f"2. 切分数据集...")
    train_df = get_dataset_by_date(full_df, TRAIN_START, TRAIN_END)
    valid_df = get_dataset_by_date(full_df, VALID_START, VALID_END)
    # ================= 样本权重（只影响训练，不影响验证） =================

    # 基础权重
    train_df['weight'] = 1.0

    # 正样本更重要（你可以从 2.0 / 3.0 开始试）
    train_df.loc[train_df[LABEL] == 1, 'weight'] = 3.0

    # 验证集权重不重要，但为了接口统一给一个
    valid_df['weight'] = 1.0

    print(f"   训练集 ({TRAIN_START} ~ {TRAIN_END}): {len(train_df)} 条")
    print(f"   验证集 ({VALID_START} ~ {VALID_END}): {len(valid_df)} 条")
    
    if train_df.empty or valid_df.empty:
        print("错误：训练集或验证集为空，请检查时间设置或数据源。")
        exit()

    # 计算基准胜率 (Base Rate)
    base_rate = valid_df[LABEL].mean()
    print(f"   验证集原始胜率 (Base Rate): {base_rate*100:.2f}%")

    # 3. 初始化训练器
    trainer = ModelTrainer(train_df, valid_df)
    results = []

    # ------------------ 模型竞技场 ------------------
    
    # Model 1: LightGBM
    lgb_model, lgb_pred = trainer.run_lightgbm()
    valid_df['score_lgb'] = lgb_pred
    acc_lgb = evaluate_top_k(valid_df, 'LightGBM', pred_col='score_lgb', top_n=5)
    results.append({'Model': 'LightGBM', 'Top5_WinRate': acc_lgb, 'Object': lgb_model})
    # Model 1b: LightGBM Rank
    lgb_rank_model, lgb_rank_pred = trainer.run_lightgbm_rank()
    valid_df['score_lgb_rank'] = lgb_rank_pred

    acc_lgb_rank = evaluate_top_k(
        valid_df,
        'LightGBM_Rank',
        pred_col='score_lgb_rank',
        top_n=8
    )

    results.append({
        'Model': 'LightGBM_Rank',
        'Top5_WinRate': acc_lgb_rank,
        'Object': lgb_rank_model
    })

    # Model 2: XGBoost
    xgb_model, xgb_pred = trainer.run_xgboost()
    valid_df['score_xgb'] = xgb_pred
    acc_xgb = evaluate_top_k(valid_df, 'XGBoost', pred_col='score_xgb', top_n=5)
    results.append({'Model': 'XGBoost', 'Top5_WinRate': acc_xgb, 'Object': xgb_model})
    
    # Model 3: CatBoost
    cat_model, cat_pred = trainer.run_catboost()
    valid_df['score_cat'] = cat_pred
    acc_cat = evaluate_top_k(valid_df, 'CatBoost', pred_col='score_cat', top_n=5)
    results.append({'Model': 'CatBoost', 'Top5_WinRate': acc_cat, 'Object': cat_model})

    # Model 4: 平均融合 (Ensemble)
    # 简单的把三个模型的分数加起来取平均，往往最稳
    valid_df['score_avg'] = (valid_df['score_lgb'] + valid_df['score_xgb'] + valid_df['score_cat']) / 3
    acc_avg = evaluate_top_k(valid_df, 'Ensemble(Avg)', pred_col='score_avg', top_n=5)
    results.append({'Model': 'Ensemble', 'Top5_WinRate': acc_avg, 'Object': None})

    # ------------------ 结果展示 ------------------
    print("\n" + "="*40)
    print("       模型竞技结果 (Top 5 命中率)")
    print("="*40)
    
    results_df = pd.DataFrame(results).sort_values(by='Top5_WinRate', ascending=False)
    
    print(f"基准胜率 (瞎买): {base_rate*100:.2f}%")
    print("-" * 40)
    for index, row in results_df.iterrows():
        lift = row['Top5_WinRate'] / base_rate
        print(f"[{row['Model']}] 胜率: {row['Top5_WinRate']*100:.2f}% | 提升度: {lift:.2f}倍")
    print("-" * 40)
    
    # 4. 保存最佳模型
    best_model_info = results_df.iloc[0]
    best_model_name = best_model_info['Model']
    
    if best_model_name == 'Ensemble':
        print("最佳策略是三个模型融合。由于代码演示原因，保存单体最强模型。")
        # 找到单体最强的
        best_single = results_df[results_df['Model'] != 'Ensemble'].iloc[0]
        best_model_obj = best_single['Object']
        best_name = best_single['Model']
    else:
        best_model_obj = best_model_info['Object']
        best_name = best_model_name
        
    save_path = os.path.join(MODEL_DIR, f"best_model_{best_name}.pkl")
    joblib.dump(best_model_obj, save_path)
    # ... (接在之前的代码后面) ...

    # ================= 6. 输出最后一天实战选股结果 =================
    print("\n" + "="*40)
    print("       🔥 最后一天实战选股模拟 🔥")
    print("="*40)
    
    # 1. 获取验证集中最后一天
    last_date = valid_df[DATE_COL].max()
    print(f"日期: {last_date.date()}")
    
    # 2. 提取当天数据
    last_day_df = valid_df[valid_df[DATE_COL] == last_date].copy()
    
    if last_day_df.empty:
        print("错误：未找到最后一天的数据。")
    else:
        # 定义一个帮助函数来展示 Top 5
        def print_top_picks(df, model_name, score_col):
            print(f"\n[{model_name}] 推荐 Top 5:")
            # 按分数降序
            top_5 = df.sort_values(by=score_col, ascending=False).head(8)
            
            # 整理输出格式
            display_cols = ['code', score_col, LABEL] # 如果 label 还是空的(因为是未来)，可能显示 NaN
            
            # 格式化输出
            print(f"{'Code':<10} | {'Score':<10} | {'Label (T+1)'}")
            print("-" * 35)
            for _, row in top_5.iterrows():
                # 转换分数为百分比显示
                score_str = f"{row[score_col]:.4f}"
                label_str = str(int(row[LABEL])) if not pd.isna(row[LABEL]) else "?"
                print(f"{row['code']:<10} | {score_str:<10} | {label_str}")

        # 3. 分别输出各模型的推荐
        # 确保列名和上面训练时保存的一致
        if 'score_cat' in last_day_df.columns:
            print_top_picks(last_day_df, "CatBoost (冠军)", 'score_cat')
            
        if 'score_lgb' in last_day_df.columns:
            print_top_picks(last_day_df, "LightGBM", 'score_lgb')
            
        if 'score_xgb' in last_day_df.columns:
            print_top_picks(last_day_df, "XGBoost", 'score_xgb')
            
        if 'score_avg' in last_day_df.columns:
            print_top_picks(last_day_df, "Ensemble (融合)", 'score_avg')

    print("\n提示：Label=1 表示第二天确实冲高成功，Label=0 表示失败。")
    print("如果这是最新的数据，Label 可能是已知的（因为是回测验证集）。")
    print(f"\n最佳单体模型 [{best_name}] 已保存至: {save_path}")
    
    # 也可以把回测数据存下来看细节
    valid_df.to_csv(os.path.join(MODEL_DIR, "model_comparison_results.csv"), index=False)

1. 读取数据: D:\work\data2\train_data.csv ...
2. 切分数据集...
   训练集 (2020-01-01 ~ 2025-01-01): 87444 条
   验证集 (2025-01-01 ~ 2025-12-31): 17563 条
   验证集原始胜率 (Base Rate): 32.77%

--- Training LightGBM ---
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[20]	valid_0's auc: 0.585329

--- Training LightGBM (LambdaRank for Top-K) ---
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@5: 0.453	valid_0's ndcg@8: 0.442876	valid_0's ndcg@10: 0.440392
[100]	valid_0's ndcg@5: 0.457347	valid_0's ndcg@8: 0.448839	valid_0's ndcg@10: 0.441666
Early stopping, best iteration is:
[66]	valid_0's ndcg@5: 0.459113	valid_0's ndcg@8: 0.451255	valid_0's ndcg@10: 0.442899

--- Training XGBoost ---

--- Training CatBoost ---

       模型竞技结果 (Top 5 命中率)
基准胜率 (瞎买): 32.77%
----------------------------------------
[CatBoost] 胜率: 45.63% | 提升度: 1.39倍
[Ensemble] 胜率: 44.24% | 提升度: 1.35倍
[LightGBM] 胜率: 44.16% | 提升度: 1.35倍
[LightGBM_Rank] 胜率: 43.24% | 提

In [20]:
train = pd.read_csv(r"D:\work\data2\train_data.csv")
train['date'] = pd.to_datetime(train['date'])
train = train[train['date'] < '2025-12-16']
train

,date,code,upper_shadow,body_size,ma5_bias,vol_ratio,high_low_ratio,limit_up_count,limit_down_count,limit_up_ratio,limit_down_ratio,avg_pct_change,label,next_open_change
0,2014-04-03,29,0.000000,0.045082,0.003445,0.899924,0.050813,10,7,0.004378,0.003065,-0.000936,1,-0.012876
1,2014-04-03,56,0.008827,0.021341,0.020884,1.280267,0.041667,10,7,0.004378,0.003065,-0.000936,0,0.002077
2,2014-04-03,541,0.019913,0.024845,0.060606,1.021527,0.057642,10,7,0.004378,0.003065,-0.000936,1,-0.019048
3,2014-04-03,582,0.004513,0.014235,0.024787,0.986297,0.028294,10,7,0.004378,0.003065,-0.000936,0,0.002708
4,2014-04-03,608,0.011186,0.028261,0.005398,1.513834,0.048936,10,7,0.004378,0.003065,-0.000936,0,-0.004474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170851,2025-12-15,605060,0.015845,0.027159,0.026012,1.670830,0.067148,64,26,0.012401,0.005038,-0.002162,0,-0.011318
170852,2025-12-15,605081,0.020036,0.015726,0.048711,1.316165,0.038961,64,26,0.012401,0.005038,-0.002162,0,-0.002732
170853,2025-12-15,605090,0.034031,0.005263,0.082521,2.000576,0.086371,64,26,0.012401,0.005038,-0.002162,0,-0.018325
170854,2025-12-15,605123,0.009372,0.063101,0.121731,1.551577,0.084337,64,26,0.012401,0.005038,-0.002162,0,-0.026608
